# 09 — Progressive Fine-tuning (Model v2)
MobileNetV3-Small — 3 asamali egitim. Dengeli veriyle pos_weight yok, over-prediction sorunu
kaynaktan cozuluyor.

| Asama | Dondurulan | Epoch | LR |
|---|---|---|---|
| 1 | Backbone tamami | 10 | 1e-3 |
| 2 | Ilk 7 blok | 20 | 3e-4 |
| 3 | Hicbir sey | 20 | 1e-4 |

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler

# --- Yerel ---
CODE_ROOT = Path('../')
DATA_ROOT = Path('../')

# --- Colab ---
# from google.colab import drive
# drive.mount('/content/drive')
# CODE_ROOT = Path('/content/drive/MyDrive/film-genre-project')
# DATA_ROOT = Path('/content/drive/MyDrive/film-genre-project-data')
# import sys; sys.path.insert(0, str(CODE_ROOT / 'src'))

import sys
sys.path.insert(0, str(CODE_ROOT / 'src'))

from dataset import PosterDataset
from transforms import train_transforms, val_transforms

POSTERS_DIR = DATA_ROOT / 'posters'
TRAIN_CSV   = DATA_ROOT / 'train.csv'
VAL_CSV     = DATA_ROOT / 'val.csv'
MLB_PKL     = DATA_ROOT / 'mlb.pkl'
CKPT_DIR    = CODE_ROOT / 'checkpoints' / 'finetune'

with open(MLB_PKL, 'rb') as f:
    mlb = pickle.load(f)
N_CLASSES = len(mlb.classes_)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  N_CLASSES: {N_CLASSES}')
print(f'Classes: {mlb.classes_.tolist()}')

## 1. Dataset ve DataLoader

In [ ]:
BATCH_SIZE  = 64
NUM_WORKERS = 2

train_ds = PosterDataset(TRAIN_CSV, POSTERS_DIR, mlb, transform=train_transforms)
val_ds   = PosterDataset(VAL_CSV,   POSTERS_DIR, mlb, transform=val_transforms)

# WeightedRandomSampler — kalan dengesizligi minibatch seviyesinde dengeleme
label_sums     = train_ds.labels.sum(axis=0)
class_weights  = 1.0 / (label_sums + 1e-6)
sample_weights = (train_ds.labels * class_weights).sum(axis=1)
sampler = WeightedRandomSampler(sample_weights.tolist(), len(sample_weights))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')

## 2. Model

In [ ]:
weights = torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
model = torchvision.models.mobilenet_v3_small(weights=weights)

# Son classifier katmanini degistir
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, N_CLASSES)

model = model.to(DEVICE)

total = sum(p.numel() for p in model.parameters())
print(f'Toplam parametre: {total:,}')
print(f'features alt-modulleri: {len(model.features)}')
for i, block in enumerate(model.features):
    n = sum(p.numel() for p in block.parameters())
    print(f'  features[{i}]: {type(block).__name__:30s} {n:>8,} param')

## 3. Yardimci Fonksiyonlar

In [ ]:
def eval_macro_f1(mdl, loader):
    mdl.eval()
    preds_all, targets_all = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits = mdl(imgs.to(DEVICE))
            preds_all.append((torch.sigmoid(logits) > 0.5).cpu().numpy())
            targets_all.append(labels.numpy())
    return f1_score(np.vstack(targets_all), np.vstack(preds_all),
                    average='macro', zero_division=0)


def run_stage(mdl, stage_cfg, history, patience=10):
    """Bir egitim asamasini calistirir, en iyi checkpoint'i kaydeder."""
    n_epochs = stage_cfg['epochs']
    ckpt     = CKPT_DIR / stage_cfg['ckpt']
    ckpt.parent.mkdir(parents=True, exist_ok=True)

    trainable = [p for p in mdl.parameters() if p.requires_grad]
    print(f"Egitilecek parametre: {sum(p.numel() for p in trainable):,}")

    criterion = nn.BCEWithLogitsLoss()  # dengeli veri — pos_weight yok
    optimizer = optim.AdamW(trainable, lr=stage_cfg['lr'], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_f1  = 0.0
    no_improve = 0

    for epoch in range(1, n_epochs + 1):
        mdl.train()
        total_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.float().to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(mdl(imgs), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        val_f1   = eval_macro_f1(mdl, val_loader)
        avg_loss = total_loss / len(train_loader)
        history['train_loss'].append(avg_loss)
        history['val_macro_f1'].append(val_f1)
        history['stage_label'].append(stage_cfg['name'])

        print(f"Epoch {epoch:2d}/{n_epochs}  loss={avg_loss:.4f}  val_macro_f1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            no_improve = 0
            torch.save(mdl.state_dict(), ckpt)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping — epoch {epoch}")
                break

    print(f">>> {stage_cfg['name']} bitti. Best val Macro F1: {best_f1:.4f}\n")
    return best_f1


history = {'train_loss': [], 'val_macro_f1': [], 'stage_label': []}
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('Hazir.')

## 4. Stage 1 — Sadece Classifier (10 epoch, lr=1e-3)

In [ ]:
# Backbone tamamen dondur
for p in model.parameters():
    p.requires_grad = False

# Sadece classifier egit
for p in model.classifier.parameters():
    p.requires_grad = True

best_s1 = run_stage(model, {'name': 'Stage1', 'epochs': 10, 'lr': 1e-3, 'ckpt': 'stage1_best.pt'}, history)

## 5. Stage 2 — Son 3 Blok + Classifier (20 epoch, lr=3e-4)

In [ ]:
# Stage 1 best'ten yukle
model.load_state_dict(torch.load(CKPT_DIR / 'stage1_best.pt', map_location=DEVICE))

# features[7], [8], [9] + classifier ac
for p in model.features[7:].parameters():
    p.requires_grad = True

best_s2 = run_stage(model, {'name': 'Stage2', 'epochs': 20, 'lr': 3e-4, 'ckpt': 'stage2_best.pt'}, history)

## 6. Stage 3 — Tam Ag (20 epoch, lr=1e-4)

In [ ]:
# Stage 2 best'ten yukle
model.load_state_dict(torch.load(CKPT_DIR / 'stage2_best.pt', map_location=DEVICE))

# Tum parametreleri ac
for p in model.parameters():
    p.requires_grad = True

best_s3 = run_stage(model, {'name': 'Stage3', 'epochs': 20, 'lr': 1e-4, 'ckpt': 'stage3_best.pt'}, history)

## 7. Egitim Egrileri

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
labels = history['stage_label']
losses = history['train_loss']
f1s    = history['val_macro_f1']

# Stage sinirlarini bul
stage_colors = {'Stage1': '#3498db', 'Stage2': '#2ecc71', 'Stage3': '#e74c3c'}
prev_stage = None
stage_boundaries = []
for i, s in enumerate(labels):
    if s != prev_stage:
        stage_boundaries.append((i + 1, s))
        prev_stage = s

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, losses, color='steelblue', linewidth=1.5)
ax1.set_title('Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')

ax2.plot(epochs, f1s, color='green', linewidth=1.5)
ax2.set_title('Val Macro F1')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Macro F1')
ax2.axhline(best_s3, linestyle='--', color='red', alpha=0.7, label=f'best={best_s3:.4f}')
ax2.legend()

# Stage gecis cizgileri
for ep, sname in stage_boundaries:
    for ax in (ax1, ax2):
        ax.axvline(ep, color='orange', linestyle=':', linewidth=1.2)
        ax.text(ep + 0.2, ax.get_ylim()[1] * 0.97, sname, fontsize=8, color='darkorange')

plt.suptitle(f'Progressive Fine-tuning — Stage 1/2/3 | Final Best F1: {best_s3:.4f}')
plt.tight_layout()
plt.show()

print(f'Stage 1 best: {best_s1:.4f}')
print(f'Stage 2 best: {best_s2:.4f}')
print(f'Stage 3 best: {best_s3:.4f}')
print(f'Checkpoint   : {CKPT_DIR}/stage3_best.pt')